In [ ]:
import re, math
import pandas as pd
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer

dokumen = [
    "Sistem komputer modern menggunakan prosesor yang sangat cepat.",
    "Jaringan komputer menghubungkan banyak perangkat dalam satu sistem.",
    "Kecerdasan buatan mampu belajar dari data secara otomatis.",
    "Sistem temu kembali informasi membantu pencarian dokumen secara cepat dan akurat."
]

stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def preprocess_simple(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return stopword_remover.remove(text).split()

tokens_per_doc = [preprocess_simple(d) for d in dokumen]
vocab = sorted(set(t for toks in tokens_per_doc for t in toks))  # semua term unik jadi kolom

# Bag-of-Words: hitung kemunculan mentah tiap term per dokumen.
# TF di sini = raw count itu sendiri.
bow = pd.DataFrame(
    [[toks.count(t) for t in vocab] for toks in tokens_per_doc],
    columns=vocab, index=[f"Dok{i+1}" for i in range(4)]
)
tf = bow.copy()

N = len(dokumen)
df = (bow > 0).sum(axis=0)          # df(t) = di berapa dokumen term itu muncul
idf = df.apply(lambda d: math.log(N / d))  # rumus IDF klasik log(N/df)

tfidf_manual = tf.mul(idf, axis=1)  # TF-IDF = TF x IDF

# Pembanding pakai scikit-learn. Default-nya pakai smooth IDF (+1 di log)
# dan normalisasi L2, jadi angkanya beda dari versi manual, tapi urutan
# term tertinggi per dokumen tetap sama.
docs_joined = [" ".join(toks) for toks in tokens_per_doc]
vectorizer = TfidfVectorizer(smooth_idf=True, norm="l2")
tfidf_sklearn = pd.DataFrame(
    vectorizer.fit_transform(docs_joined).toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Dok{i+1}" for i in range(4)]
)

print(idf.round(4))
print(tfidf_manual.round(4))
print(tfidf_sklearn.round(4))

# term dengan TF-IDF tertinggi per dokumen
for idx in tfidf_manual.index:
    print(idx, tfidf_manual.loc[idx].idxmax(), round(tfidf_manual.loc[idx].max(), 4))


#ANALISIS: Hasil pembobotan menunjukkan bahwa TF-IDF memberikan bobot lebih tinggi pada term yang lebih spesifik 
# dan tidak banyak muncul di dokumen lain. Perhitungan manual menggunakan rumus TF × IDF, 
# sedangkan TfidfVectorizer menggunakan normalisasi dan smooth IDF, sehingga nilai keduanya dapat berbeda. 
# Meskipun demikian, keduanya digunakan untuk menunjukkan tingkat kepentingan suatu term dalam setiap dokumen.


akurat           1.3863
banyak           1.3863
belajar          1.3863
buatan           1.3863
cepat            0.6931
data             1.3863
dokumen          1.3863
informasi        1.3863
jaringan         1.3863
kecerdasan       1.3863
komputer         0.6931
mampu            1.3863
membantu         1.3863
menggunakan      1.3863
menghubungkan    1.3863
modern           1.3863
otomatis         1.3863
pencarian        1.3863
perangkat        1.3863
prosesor         1.3863
sangat           1.3863
satu             1.3863
sistem           0.2877
temu             1.3863
dtype: float64
      akurat  banyak  belajar  buatan   cepat    data  dokumen  informasi  \
Dok1  0.0000  0.0000   0.0000  0.0000  0.6931  0.0000   0.0000     0.0000   
Dok2  0.0000  1.3863   0.0000  0.0000  0.0000  0.0000   0.0000     0.0000   
Dok3  0.0000  0.0000   1.3863  1.3863  0.0000  1.3863   0.0000     0.0000   
Dok4  1.3863  0.0000   0.0000  0.0000  0.6931  0.0000   1.3863     1.3863   

      jaringan  kecerda